In [2]:
%%writefile optimize.l
%{
#include "optimize.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%%

[a-zA-Z][a-zA-Z0-9]*    {
                            yylval.str = strdup(yytext);
                            return ID;
                         }

[0-9]+                  {
                            yylval.str = strdup(yytext);
                            return NUM;
                         }

"="                     { return '='; }
"+"                     { return '+'; }
"-"                     { return '-'; }
"*"                     { return '*'; }
"/"                     { return '/'; }
";"                     { return ';'; }

[ \t\n]                 { /* Ignore whitespace */ }

.                       { return yytext[0]; }

%%

int yywrap(void)
{
    return 1;
}

Writing optimize.l


In [3]:
%%writefile optimize.y
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>
#include <ctype.h>

int yylex(void);
int yyerror(const char *s);
%}

%union {
    char *str;
}

%token <str> ID NUM
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt_list
    : stmt_list stmt
    | stmt
    ;

stmt
    : ID '=' expr ';'
      {
          printf("%s = %s\n", $1, $3);
          free($1);
          free($3);
      }
    ;

expr
    : NUM
      {
          $$ = $1;
      }

    | ID
      {
          $$ = $1;
      }

    | expr '+' expr
      {
          if (isdigit((unsigned char)$1[0]) &&
              isdigit((unsigned char)$3[0]))
          {
              char buf[50];

              sprintf(buf, "%d", atoi($1) + atoi($3));
              $$ = strdup(buf);

              printf("// Constant Folding: %s + %s -> %s\n",
                     $1, $3, $$);
          }
          else if (strcmp($3, "0") == 0)
          {
              $$ = strdup($1);

              printf("// Algebraic Simplification: x + 0 -> x\n");
          }
          else if (strcmp($1, "0") == 0)
          {
              $$ = strdup($3);

              printf("// Algebraic Simplification: 0 + x -> x\n");
          }
          else
          {
              char buf[100];

              sprintf(buf, "%s + %s", $1, $3);
              $$ = strdup(buf);
          }

          free($1);
          free($3);
      }

    | expr '-' expr
      {
          if (isdigit((unsigned char)$1[0]) &&
              isdigit((unsigned char)$3[0]))
          {
              char buf[50];

              sprintf(buf, "%d", atoi($1) - atoi($3));
              $$ = strdup(buf);

              printf("// Constant Folding: %s - %s -> %s\n",
                     $1, $3, $$);
          }
          else if (strcmp($3, "0") == 0)
          {
              $$ = strdup($1);

              printf("// Algebraic Simplification: x - 0 -> x\n");
          }
          else
          {
              char buf[100];

              sprintf(buf, "%s - %s", $1, $3);
              $$ = strdup(buf);
          }

          free($1);
          free($3);
      }

    | expr '*' expr
      {
          if (isdigit((unsigned char)$1[0]) &&
              isdigit((unsigned char)$3[0]))
          {
              char buf[50];

              sprintf(buf, "%d", atoi($1) * atoi($3));
              $$ = strdup(buf);

              printf("// Constant Folding: %s * %s -> %s\n",
                     $1, $3, $$);
          }
          else if (strcmp($3, "1") == 0)
          {
              $$ = strdup($1);

              printf("// Algebraic Simplification: x * 1 -> x\n");
          }
          else if (strcmp($3, "2") == 0)
          {
              char buf[100];

              sprintf(buf, "%s + %s", $1, $1);
              $$ = strdup(buf);

              printf("// Strength Reduction: x * 2 -> x + x\n");
          }
          else
          {
              char buf[100];

              sprintf(buf, "%s * %s", $1, $3);
              $$ = strdup(buf);
          }

          free($1);
          free($3);
      }

    | expr '/' expr
      {
          if (isdigit((unsigned char)$1[0]) &&
              isdigit((unsigned char)$3[0]))
          {
              if (atoi($3) == 0)
              {
                  yyerror("Division by zero");
                  $$ = strdup("0");
              }
              else
              {
                  char buf[50];

                  sprintf(buf, "%d", atoi($1) / atoi($3));
                  $$ = strdup(buf);

                  printf("// Constant Folding: %s / %s -> %s\n",
                         $1, $3, $$);
              }
          }
          else if (strcmp($3, "1") == 0)
          {
              $$ = strdup($1);

              printf("// Algebraic Simplification: x / 1 -> x\n");
          }
          else
          {
              char buf[100];

              sprintf(buf, "%s / %s", $1, $3);
              $$ = strdup(buf);
          }

          free($1);
          free($3);
      }
    ;

%%

int main(void)
{
    printf("Enter Three Address Code statements:\n");
    printf("Example: a = 2 + 3;\n");
    printf("Press Ctrl+D when finished.\n\n");

    yyparse();

    return 0;
}

int yyerror(const char *s)
{
    fprintf(stderr, "Syntax Error: %s\n", s);
    return 0;
}

Writing optimize.y


In [4]:
!bison -d optimize.y
!flex optimize.l

In [5]:
!gcc optimize.tab.c lex.yy.c -o optimize

In [6]:
!echo "a = 2 + 3;" | ./optimize

Enter Three Address Code statements:
Example: a = 2 + 3;
Press Ctrl+D when finished.

// Constant Folding: 2 + 3 -> 5
a = 5
